# H33 Partial Qwen Kaggle Smoke

Controlled smoke training on selected **original** `Qwen/Qwen2.5-0.5B-Instruct` weights. This notebook does not install/replace Torch or Transformers before inspecting Kaggle's environment. Public Kaggle docker-python release v170 uses `transformers>=5.0.0`; the trainer supports both Transformers 4.x and 5.x dtype APIs.

Default path: environment snapshot → preference preparation → 2 optimizer steps → exact resume to step 3 → evaluation gate. INT4 export is opt-in only after evaluation passes.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, glob

REPO_URL = 'https://github.com/Musabgpt/H33.git'
BRANCH = 'h33-partial-train-native'
ROOT = Path('/kaggle/working/H33')
WORK = Path('/kaggle/working/h33-partial-run')
PREP = WORK / 'prepared'
TRAIN_OUT = WORK / 'train'
WORK.mkdir(parents=True, exist_ok=True)
print({'root': str(ROOT), 'work': str(WORK)})


In [ ]:
# Clone/update only the experiment branch. Internet must be enabled because the base Qwen model is fetched from Hugging Face.
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(ROOT)], check=True)
else:
    subprocess.run(['git','-C',str(ROOT),'fetch','origin',BRANCH,'--depth','1'], check=True)
    subprocess.run(['git','-C',str(ROOT),'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',str(ROOT),'reset','--hard',f'origin/{BRANCH}'], check=True)
os.chdir(ROOT)
print(subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())


In [ ]:
# M1 preflight: record the actual Kaggle runtime BEFORE changing dependencies.
env_json = WORK / 'h33-environment.json'
freeze_txt = WORK / 'h33-kaggle-freeze.txt'
with env_json.open('w', encoding='utf-8') as out:
    subprocess.run([sys.executable,'training/check_environment.py'], stdout=out, check=True)
with freeze_txt.open('w', encoding='utf-8') as out:
    subprocess.run([sys.executable,'-m','pip','freeze'], stdout=out, check=True)
report = json.loads(env_json.read_text(encoding='utf-8'))
print(json.dumps(report, ensure_ascii=False, indent=2))
assert report.get('cuda_available'), 'GPU/CUDA is required for the real-weight smoke run'
assert report.get('packages',{}).get('torch'), 'Torch missing from Kaggle runtime'
assert report.get('packages',{}).get('transformers'), 'Transformers missing from Kaggle runtime'


In [ ]:
# Auto-discover H33 exports. Override SFT_FILES / PREF_FILES here if your Kaggle dataset uses other names.
sft_candidates = sorted(Path('/kaggle/input').rglob('H33_SFT*.jsonl'))
pref_candidates = sorted(Path('/kaggle/input').rglob('H33_DPO*.jsonl'))
SFT_FILES = sft_candidates
PREF_FILES = pref_candidates
print('SFT:', [str(p) for p in SFT_FILES])
print('DPO:', [str(p) for p in PREF_FILES])
assert SFT_FILES, 'No H33_SFT*.jsonl found under /kaggle/input'


In [ ]:
# Deterministic collapse/split of latest effective decisions.
cmd = [sys.executable,'training/prepare_preferences.py']
for p in SFT_FILES: cmd += ['--sft', str(p)]
for p in PREF_FILES: cmd += ['--preferences', str(p)]
cmd += ['--output-dir', str(PREP), '--seed', '3407']
subprocess.run(cmd, check=True)
prep_report = json.loads((PREP/'prepare_report.json').read_text(encoding='utf-8'))
print(json.dumps(prep_report, ensure_ascii=False, indent=2))
assert prep_report['train_turns'] >= 1


In [ ]:
# Real original-weight smoke: two completed optimizer steps, checkpoint every step.
subprocess.run([
    sys.executable,'training/train_partial_sft.py',
    '--train-sft', str(PREP/'train_sft.jsonl'),
    '--output-dir', str(TRAIN_OUT),
    '--epochs','50',
    '--gradient-accumulation-steps','1',
    '--checkpoint-every-steps','1',
    '--max-optimizer-steps','2',
    '--max-length','256',
    '--seed','3407',
], check=True)
summary = json.loads((TRAIN_OUT/'training_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['optimizer_steps'] == 2
assert summary['original_weights_changed'] is True


In [ ]:
# Exact resume smoke: continue from checkpoint-00000002 to global optimizer step 3.
ckpt = TRAIN_OUT / 'checkpoint-00000002'
assert ckpt.is_dir(), ckpt
subprocess.run([
    sys.executable,'training/train_partial_sft.py',
    '--train-sft', str(PREP/'train_sft.jsonl'),
    '--output-dir', str(TRAIN_OUT),
    '--resume-from-checkpoint', str(ckpt),
    '--epochs','50',
    '--gradient-accumulation-steps','1',
    '--checkpoint-every-steps','1',
    '--max-optimizer-steps','3',
    '--max-length','256',
    '--seed','3407',
], check=True)
summary = json.loads((TRAIN_OUT/'training_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))
assert summary['starting_optimizer_step'] == 2
assert summary['optimizer_steps'] == 3
assert summary['original_weights_changed'] is True


In [ ]:
# Regression gate. This may intentionally fail (exit 2) if a 3-step smoke checkpoint regresses; capture the report either way.
eval_report = TRAIN_OUT / 'evaluation_report.json'
proc = subprocess.run([
    sys.executable,'training/evaluate_partial.py',
    '--base-model','Qwen/Qwen2.5-0.5B-Instruct',
    '--trained-model', str(TRAIN_OUT/'final'),
    '--eval', str(PREP/'eval.jsonl'),
    '--output', str(eval_report),
    '--max-length','256',
    '--max-new-tokens','64',
])
evaluation = json.loads(eval_report.read_text(encoding='utf-8'))
print(json.dumps({k:evaluation[k] for k in [
    'baseline_general_pass_rate','trained_general_pass_rate',
    'baseline_preference_margin','trained_preference_margin',
    'preference_pairs','passes_gate']}, ensure_ascii=False, indent=2))
print('evaluator_exit_code=', proc.returncode)


In [ ]:
# Optional INT4 export. Leave False for the first smoke. It is blocked unless evaluation + original-weight proof pass.
EXPORT_INT4 = False
if EXPORT_INT4:
    try:
        import onnxruntime_genai  # noqa
    except Exception:
        raise RuntimeError('onnxruntime-genai is missing. Inspect the saved freeze first; install a compatible version deliberately, then rerun this cell.')
    subprocess.run([
        sys.executable,'training/export_mobile_int4.py',
        '--trained-model', str(TRAIN_OUT/'final'),
        '--training-summary', str(TRAIN_OUT/'training_summary.json'),
        '--evaluation-report', str(eval_report),
        '--output-dir', str(WORK/'int4'),
    ], check=True)


In [ ]:
# Compact final evidence bundle paths.
for p in [env_json, freeze_txt, PREP/'prepare_report.json', TRAIN_OUT/'training_summary.json', TRAIN_OUT/'evaluation_report.json']:
    print(p, 'OK' if p.exists() else 'MISSING')
